# LLMPirate Lab Report  
### Netlist rewriting, text similarity, and formal equivalence

This report documents experiments from the LLMPirate-style workflow: rewriting a gate-level benchmark so that a **text similarity** score drops below **0.3** while (ideally) preserving **combinational behavior**, checked with **Yosys** (miter + SAT).

**Benchmark:** `C432` / variant `c432-CS320` (combinational `top` module, **212** gates in the provided netlist).  
**Success criterion (lab):** SIM \< **0.3** *and* formal equivalence **proven** (or at least no counterexample).  
**Platform note:** On macOS the notebook often uses a Python **difflib** similarity proxy when the Linux `sim_text` binary cannot run; **Yosys results are still the primary functional ground truth.**

**Tasks covered**
1. Direct full-netlist Verilog rewrite (Task 1)  
2. Boolean-equation intermediate rewrite (Task 2)  
3. Divide-and-conquer gate-type mapping without feedback (Task 3)  
4. Divide-and-conquer with iterative feedback (Task 4)  
5. Comparison against cached research mappings (Task 5)

After you run the pipeline, use the **Inference — recorded run** table at the end of the notebook (and the per-task **Results and discussion** tables) for your write-up — refresh those sections when you change model, circuit, or environment.

---

## How to reproduce

Execute code cells **0.1 → 0.3** once, then run Tasks **1–5** in order. API access is via Portkey to the NYU OpenAI-compatible gateway. Cell **0.3** loads `.env` from the project folder or parents. In cell **0.2**, **`NOTEBOOK_SKIP_SLOW_LLM_TASKS`** (default **True**) skips slow **Task 1** and **Task 2** LLM calls; set it to **False** for full direct rewrites. For a **fast** NOR pipeline, keep the skip flag on and rely on **Task 3** (canonical NOR). All quantitative rows below are tied to the saved `LLM_IP_assignment/tmp_eval/*` artifacts produced by those cells.


## Environment

Run **0.1**, **0.2**, and **0.3** before any task.

| Requirement | Purpose |
|-------------|---------|
| `OPENAI_API_KEY` or `NYU_GATEWAY_API_KEY` | LLM calls (Portkey client); loaded from env or `.env` (see cell 0.3) |
| `NOTEBOOK_SKIP_SLOW_LLM_TASKS` in cell **0.2** | `True` = skip Task 1 & 2 LLM (fast); `False` = run those experiments |
| `yosys` on `PATH` | Formal equivalence (`miter` + `sat`) |
| Linux / Colab (optional) | Native `sim_text` binary; otherwise difflib fallback |

The assignment data live under `LLM_IP_assignment/` (cloned by cell 0.1 or used in-place when `src/` and `sim/` already exist).


In [1]:
#@title 0.1 Clone Repository & Install Dependencies
import os
import sys
import shutil
import subprocess

# --- Set the data directory (works in Colab and local Jupyter) ---
if os.path.isdir('/content') and os.access('/content', os.W_OK):
    DATA_DIR = '/content/LLM_IP_assignment'
else:
    _here = os.path.abspath(os.getcwd())
    _nested = os.path.abspath('LLM_IP_assignment')
    if os.path.isdir(os.path.join(_here, 'src')) and os.path.isdir(os.path.join(_here, 'sim')):
        DATA_DIR = _here
    elif os.path.isdir(_nested):
        DATA_DIR = _nested
    else:
        DATA_DIR = _nested

if not os.path.exists(DATA_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/matthewdelorenzo/LLM_IP_assignment.git', DATA_DIR],
        check=False,
    )

os.chdir(DATA_DIR)
print('Working directory:', os.getcwd())

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'networkx', 'openai', 'portkey-ai', 'python-dotenv'],
    check=False,
)

# Yosys (formal equivalence): apt on Colab/Debian; Homebrew on macOS
if shutil.which('yosys'):
    print('Yosys found:', shutil.which('yosys'))
elif sys.platform.startswith('linux'):
    subprocess.run(['apt-get', 'install', '-y', '-q', 'yosys'], check=False)
elif sys.platform == 'darwin' and shutil.which('brew'):
    print('Installing Yosys via Homebrew...')
    subprocess.run(['brew', 'install', 'yosys'], check=False)
else:
    print('Yosys not found. For formal equivalence checks install it:')
    print('  macOS:  brew install yosys')
    print('  Debian/Ubuntu:  sudo apt-get install -y yosys')

# SIM text tool: upstream ships a Linux x86_64 ELF binary only
SIM_EXECUTABLE = os.path.join(DATA_DIR, 'sim', 'sim_text')
if os.path.exists(SIM_EXECUTABLE):
    os.chmod(SIM_EXECUTABLE, 0o755)
    with open(SIM_EXECUTABLE, 'rb') as _bf:
        _magic = _bf.read(4)
    if _magic == b'\x7fELF' and not sys.platform.startswith('linux'):
        print('NOTICE: sim_text is a Linux-only binary — it will not run on', sys.platform)
        print('        Use Google Colab / Linux, or the notebook will use a Python similarity fallback.')
    else:
        print('SIM binary ready.')
else:
    print('WARNING: SIM binary not found at', SIM_EXECUTABLE)


Working directory: /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW6_LLMPirate/LLM_IP_assignment
Yosys found: /opt/homebrew/bin/yosys
NOTICE: sim_text is a Linux-only binary — it will not run on darwin
        Use Google Colab / Linux, or the notebook will use a Python similarity fallback.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
#@title 0.2 Imports and Helper Functions
import sys, os, glob, pickle, copy, random, subprocess, time, re, difflib
import networkx as nx
import openai

# Add src/ directories to path
sys.path.insert(0, os.path.join(DATA_DIR, 'src'))
sys.path.insert(0, os.path.join(DATA_DIR, 'src_ablation_study_wo_SolB'))

# Core functions from original codebase
from characterize_circuit import (
    characterize_generic_netlist,
    create_LLM_circuit_query_template,
    is_gate_line,
    convert_circuit_to_Boolean_format,
    convert_circuit_from_Boolean_format,
)
from circuit_decomposer import (
    parse_expression,
    decompose_expression,
    final_decompose_expression,
)
from sim_evaluation import evaluate_sim_text
from config import DATASET_PATH, SIM_DIR


def _python_text_similarity(a, b):
    """Fallback when SIM binary cannot run; same scale (higher = more similar)."""
    return difflib.SequenceMatcher(None, a, b).ratio()


def evaluate_sim(orig_contents, pirated_contents, label='test'):
    """Save circuits to temp files and run SIM text comparison."""
    os.makedirs('tmp_eval', exist_ok=True)
    orig_path = f'tmp_eval/orig_{label}.v'
    pirated_path = f'tmp_eval/pirated_{label}.v'

    pirated_clean = _prepare_verilog_for_yosys(
        _fix_verilog_declarations(_extract_first_verilog_module(pirated_contents))
    )

    with open(orig_path, 'w') as f:
        f.write(orig_contents)
    with open(pirated_path, 'w') as f:
        f.write(pirated_clean)

    try:
        result = evaluate_sim_text(orig_path, pirated_path)
        if result.returncode != 0:
            print(f'SIM error (exit code {result.returncode}): {result.stderr}')
            return None
        if 'consists for' in result.stdout:
            score = float(
                result.stdout.split('\n')[-2]
                .split('consists for')[-1]
                .split('%')[0].strip()
            ) / 100.0
            return score
        return 0.0
    except Exception as e:
        estr = str(e).lower()
        errno = getattr(e, 'errno', None)
        if errno == 8 or 'exec format' in estr:
            if not getattr(evaluate_sim, '_fallback_warned', False):
                print(
                    'SIM: native binary not runnable on this OS — using difflib similarity '
                    '(use Colab/Linux for the real SIM tool).'
                )
                evaluate_sim._fallback_warned = True
            return _python_text_similarity(orig_contents, pirated_clean)
        print(f'SIM error: {e}')
        return None


def _needs_verilog_continuation(text):
    """True if fenced/plain Verilog looks started but not finished (no endmodule)."""
    if not text:
        return False
    if re.search(r'(?is)\bendmodule\b', text):
        return False
    t = _strip_code_fences(text)
    return bool(re.search(r'(?is)\bmodule\b', t)) and not re.search(r'(?is)\bendmodule\b', t)


def _strip_code_fences(text):
    """Remove markdown code fences; merge ALL ``` ``` blocks (LLMs often close/reopen mid-netlist)."""
    if '```' not in text:
        return text
    parts = text.split('```')
    if len(parts) < 2:
        return text
    _LANG = {'', 'verilog', 'v', 'sv', 'systemverilog'}
    _SKIP_LANG = {'python', 'py', 'bash', 'sh', 'json', 'markdown', 'md', 'text', 'plaintext'}
    chunks = []
    for i in range(1, len(parts), 2):
        block = parts[i]
        lines = block.split('\n')
        head = lines[0].strip().lower() if lines else ''
        if head in _SKIP_LANG:
            continue
        if lines and head in _LANG:
            block = '\n'.join(lines[1:])
        chunks.append(block)
    merged = '\n'.join(c for c in chunks if c is not None)
    return merged.strip() if merged.strip() else text


def _extract_first_verilog_module(text):
    """Strip fences, then take first module ... endmodule (LLMs often add commentary)."""
    text = _strip_code_fences(text)
    m = re.search(r'(?is)\bmodule\s+\w+.*?endmodule\b', text)
    if m:
        return m.group(0).strip()
    for blk in re.finditer(r'```(?:verilog|v|sv|systemverilog)?\s*([\s\S]*?)```', text, re.I):
        inner = blk.group(1).strip()
        m = re.search(r'(?is)\bmodule\s+\w+.*?endmodule\b', inner)
        if m:
            return m.group(0).strip()
    m = re.search(r'(?is)\bmodule\s+\w+', text)
    if m:
        start = m.start()
        ends = list(re.finditer(r'(?is)\bendmodule\b', text[start:]))
        if ends:
            return text[start : start + ends[-1].end()].strip()
    return text.strip()


def _fix_verilog_declarations(verilog):
    """
    Fix common LLM Verilog output issues:
    - Add missing semicolons to input/output/wire/reg declaration lines
      that end a statement but are missing the terminating ';'
    """
    _DECL_KW = re.compile(r'^\s*(input|output|inout|wire|reg)\b')
    fixed_lines = []
    lines = verilog.splitlines()
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        if _DECL_KW.match(stripped):
            # Only add ';' if the line doesn't already end with ';' or ','
            # and it's not a continuation line (next line also starts a declaration
            # or is blank / endmodule)
            if stripped and not stripped.endswith(';') and not stripped.endswith(','):
                stripped = stripped + ';'
        fixed_lines.append(stripped)
    return '\n'.join(fixed_lines)


def _prepare_verilog_for_yosys(verilog):
    """
    Repair common LLM / Boolean-converter issues before Yosys read_verilog:

    - Illegal ';' before ')' inside ANSI port lists (LLMs often terminate the last port with ';').
    - Missing ');' after `module foo (...)` when the next line starts with input/output/inout
      (Verilog-1995 style lists ports in parentheses, then declares directions — the closing
      ');' is still required before those lines).
    - Line wraps that split an identifier from a trailing index (e.g. `net_w` newline `1,` → `net_w1,`).
      Otherwise Yosys may report errors like "1 is not a sized constant" at the broken line.
    """
    v = re.sub(
        r"([A-Za-z_]\w*)\s*\n\s*(\d+)(\s*[,);])",
        r"\1\2\3",
        verilog,
    )
    v = re.sub(r';(?=\s*\n\s*\))', '', v)
    m = re.search(r'\bmodule\s+(\w+)\s*\(', v)
    if not m:
        return v
    open_paren = m.end() - 1
    depth = 0
    close_paren = None
    for k in range(open_paren, len(v)):
        if v[k] == '(':
            depth += 1
        elif v[k] == ')':
            depth -= 1
            if depth == 0:
                close_paren = k
                break
    if close_paren is None:
        return v
    suf = v[close_paren + 1 :]
    if re.match(r'^\s*\n\s*(?:input|output|inout)\b', suf) and not re.match(r'^\s*;', suf):
        v = v[: close_paren + 1] + ';' + suf
    return v


def _get_module_name(verilog):
    """Extract the top module name from a Verilog string."""
    for line in verilog.splitlines():
        m = re.match(r'\s*module\s+(\w+)', line)
        if m:
            return m.group(1)
    return None


def check_functional_equivalence(orig_verilog, transformed_verilog, label='test'):
    """
    Formal equivalence check using Yosys miter circuit + SAT solver.

    Runs the following Yosys flow:
        read_verilog <truth>
        read_verilog <generated>
        prep; proc; opt; memory;
        clk2fflogic;
        miter -equiv -flatten <gate> <gold> miter
        sat -seq 50 -verify -prove trigger 0 -show-all \
            -show-inputs -show-outputs -set-init-zero miter

    Returns: (is_equivalent: bool | None, details: str)
        True  — formally proven equivalent
        False — counterexample found (not equivalent)
        None  — Yosys error or inconclusive result
    """
    os.makedirs('tmp_eval', exist_ok=True)

    # Strip markdown / commentary; fix declarations; repair Yosys-parse issues (LLM/converter)
    orig_clean = _prepare_verilog_for_yosys(
        _fix_verilog_declarations(_extract_first_verilog_module(orig_verilog))
    )
    trans_clean = _prepare_verilog_for_yosys(
        _fix_verilog_declarations(_extract_first_verilog_module(transformed_verilog))
    )

    orig_module = _get_module_name(orig_clean)
    gen_module  = _get_module_name(trans_clean)

    if not orig_module:
        return None, "Could not extract module name from original Verilog"
    if not gen_module:
        return None, (
            "Could not extract module name from transformed Verilog — the model must output "
            "a complete `module ... endmodule` block (no prose-only answers)."
        )
    if not re.search(r'(?is)\bendmodule\b', trans_clean):
        return None, (
            "Transformed Verilog is incomplete (missing `endmodule`) — response was likely truncated. "
            "Raise max_tokens / continuation limits for direct Verilog runs, or use divide-and-conquer tasks."
        )

    # Rename both modules to unique names so Yosys never sees a name clash
    gold_name = 'gold_circuit'
    gate_name = 'gate_circuit'

    gold_verilog = re.sub(
        r'\bmodule\s+' + re.escape(orig_module) + r'\b',
        f'module {gold_name}', orig_clean, count=1
    )
    gate_verilog = re.sub(
        r'\bmodule\s+' + re.escape(gen_module) + r'\b',
        f'module {gate_name}', trans_clean, count=1
    )

    truth_path  = os.path.abspath(f'tmp_eval/truth_{label}.v')
    gen_path    = os.path.abspath(f'tmp_eval/gen_{label}.v')
    script_path = os.path.abspath(f'tmp_eval/eq_{label}.ys')

    with open(truth_path, 'w') as f:
        f.write(gold_verilog)
    with open(gen_path, 'w') as f:
        f.write(gate_verilog)

    yosys_script = (
        f"read_verilog {truth_path}\n"
        f"read_verilog {gen_path}\n"
        "prep; proc; opt; memory;\n"
        "clk2fflogic;\n"
        f"miter -equiv -flatten {gate_name} {gold_name} miter\n"
        "sat -seq 50 -verify -prove trigger 0 "
        "-show-all -show-inputs -show-outputs -set-init-zero miter\n"
    )
    with open(script_path, 'w') as f:
        f.write(yosys_script)

    print(f"🔍 Running Yosys equivalence check for '{label}'...")
    try:
        result = subprocess.run(
            ['yosys', '-s', script_path],
            capture_output=True, text=True, timeout=180
        )
        output = result.stdout + result.stderr

        if 'no model found: SUCCESS' in output:
            return True, "Circuits formally verified equivalent (Yosys miter+SAT)"
        elif 'proof did fail' in output:
            return (
                False,
                "Circuits NOT equivalent — Yosys SAT found a counterexample. "
                "Gate-replacement pipelines (cached or LLM) can fail if a mapping is wrong or "
                "replacements interact badly. Try another strategy, fix templates, or verify with simulation.",
            )
        elif result.returncode != 0:
            err_lines = [l.strip() for l in output.splitlines() if 'ERROR' in l or 'Error' in l]
            err_msg = '; '.join(err_lines[:3]) if err_lines else output[-400:]
            return None, f"Yosys error: {err_msg}"
        else:
            return None, f"Yosys result inconclusive (exit {result.returncode})"

    except subprocess.TimeoutExpired:
        return None, "Yosys equivalence check timed out (180 s)"
    except FileNotFoundError:
        return None, (
            "Yosys not found — install: macOS `brew install yosys`, "
            "Debian/Ubuntu `sudo apt-get install -y yosys`, then re-run the setup cell."
        )
    except Exception as e:
        return None, f"Equivalence check error: {e}"


def get_mapped_circuit(orig_file_contents, cached_circuit_mapping, mapping_strategy, rank=0):
    """
    Replace every gate in the original circuit with the LLM's cached rephrase.
    From evaluate_piracy_using_cached_mapping_multiprocessing_v2.py in the original codebase.
    """
    new_lines = []
    new_wires = []
    lines = orig_file_contents.split(';')
    lines = [line.strip() for line in lines if line.strip()]
    lines = [line.replace('\n', ' ') for line in lines]
    for line in lines:
        if line.startswith(('module ', 'input ', 'output ', 'wire ', 'endmodule')):
            new_lines.append(line)
            continue
        if is_gate_line(line):
            keep_line_same = False
            gate_type = line.split()[0] + '_' + str(line.count(','))
            if gate_type not in cached_circuit_mapping or len(cached_circuit_mapping[gate_type]) == 0:
                new_lines.append(line)
                continue
            if len(list(cached_circuit_mapping[gate_type].keys())) > 0:
                line = line.replace('(', ' ( ')
                line = line.replace(')', ' ) ')
                gate_type = line.split(' ')[0] + '_' + str(line.count(','))
                gate_name = line.split()[1].strip()
                op = line.split()[3].strip(',').strip()
                num_ips = line.count(',')
                ips = [line.split()[4+j].strip().strip(',').strip() for j in range(num_ips)]
                _map_keys = list(cached_circuit_mapping[gate_type].keys())
                if mapping_strategy == 'random':
                    target_mapping_gate = random.choice(_map_keys)
                elif mapping_strategy in ('deterministic', 'sorted', 'first'):
                    target_mapping_gate = sorted(_map_keys)[0]
                else:
                    if mapping_strategy.split('_')[0] == gate_type.split('_')[0].upper():
                        keep_line_same = True
                    elif mapping_strategy in _map_keys:
                        target_mapping_gate = copy.deepcopy(mapping_strategy)
                    else:
                        target_mapping_gate = sorted(_map_keys)[0]
                if keep_line_same:
                    new_lines.append(line)
                else:
                    mapped_circuit_str = cached_circuit_mapping[gate_type][target_mapping_gate][0]
                    nets = []
                    mapped_lines = mapped_circuit_str.split('\n')
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_output = l.split('=')[0].strip()
                        nets.extend(tmp_inputs)
                        nets.append(tmp_output)
                    nets = sorted({n for n in nets if n}, key=len, reverse=True)
                    inter_net_cnt = 0
                    for net in nets:
                        if net == 'Y':
                            for pat in ['Y = ', 'Y= ', 'Y =', 'Y=']:
                                if pat in mapped_circuit_str:
                                    mapped_circuit_str = mapped_circuit_str.replace(pat, op + pat[1:])
                                    break
                        elif net in ['A' + str(i) for i in range(1, 50)]:
                            idx = int(net[1:]) - 1
                            for pat in ['(' + net + ',', ' ' + net + ',', ' ' + net + ')', '(' + net + ')']:
                                repl = pat.replace(net, ips[idx])
                                mapped_circuit_str = mapped_circuit_str.replace(pat, repl)
                        else:
                            wire_name = gate_name + '_inter_net_' + str(inter_net_cnt)
                            inter_net_cnt += 1
                            mapped_circuit_str = re.sub(
                                r'(?<![A-Za-z0-9_$])' + re.escape(net) + r'(?![A-Za-z0-9_$])',
                                wire_name,
                                mapped_circuit_str,
                            )
                            new_wires.append(wire_name)
                    mapped_lines = mapped_circuit_str.split('\n')
                    cnt = 0
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        if not l:
                            continue
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_inputs = [x for x in tmp_inputs if x != '']
                        tmp_output = l.split('=')[0].strip()
                        rhs = l.split('=', 1)[1].strip() if '=' in l else ''
                        _gm = re.search(
                            r'\b(NOT|AND|OR|NAND|NOR|XOR|XNOR)\b\s*\(', rhs, re.I
                        )
                        tmp_gate_type = (
                            _gm.group(1).upper()
                            if _gm
                            else l.split('=')[-1].split('(')[0].strip().upper()
                        )
                        if not tmp_gate_type.strip():
                            continue
                        if tmp_gate_type == 'NOT':
                            if not tmp_inputs:
                                tmp_inputs = ["1'b0"]
                            else:
                                tmp_inputs = tmp_inputs[:1]
                        elif tmp_gate_type in ('AND', 'OR', 'NAND', 'NOR', 'XOR', 'XNOR'):
                            if len(tmp_inputs) >= 2:
                                tmp_inputs = tmp_inputs[:2]
                            elif len(tmp_inputs) == 1:
                                tmp_inputs = [tmp_inputs[0], tmp_inputs[0]]
                            else:
                                tmp_inputs = ["1'b0", "1'b0"]
                        _gn = re.sub(r'[^A-Za-z0-9_$]', '_', gate_name)
                        _inst = 'g_' + _gn + '_' + str(cnt)
                        new_str = (
                            tmp_gate_type.lower()
                            + ' '
                            + _inst
                            + ' ( '
                            + tmp_output
                            + ', '
                            + ', '.join(tmp_inputs)
                            + ' )'
                        )
                        cnt += 1
                        new_lines.append(new_str)
            else:
                new_lines.append(line)
        else:
            new_lines.append(line)
    if new_wires:
        for i in range(len(new_lines)):
            if new_lines[i].split()[0] == 'wire':
                new_lines[i] = new_lines[i] + ';\nwire ' + ', '.join(new_wires)
                break
    return ';\n'.join(new_lines)


def _nor_dup(x):
    return f"NOR({x}, {x})"


def _canonical_nor_rhs(gate_type):
    """
    Return a single nested-NOR expression equivalent to the LLM template gate (Y = OP(A1..An)).
    Used when Tasks 3/4 target only NOR — avoids incorrect LLM Boolean algebra.
    """
    try:
        kind, ns = gate_type.rsplit("_", 1)
        n = int(ns)
    except ValueError:
        return None
    vs = [f"A{i+1}" for i in range(n)]

    def AND2e(e1, e2):
        return f"NOR({_nor_dup(e1)}, {_nor_dup(e2)})"

    def ANDn(v):
        v = list(v)
        acc = AND2e(v[0], v[1])
        for x in v[2:]:
            acc = AND2e(acc, x)
        return acc

    def NANDn(v):
        a = ANDn(v)
        return f"NOR({a}, {a})"

    def OR2e(e1, e2):
        return f"NOR(NOR({e1}, {e2}), NOR({e1}, {e2}))"

    def ORn(v):
        v = list(v)
        acc = OR2e(v[0], v[1])
        for x in v[2:]:
            acc = OR2e(acc, x)
        return acc

    def NORns(v):
        o = ORn(v)
        return f"NOR({o}, {o})"

    _t1 = f"NOR({_nor_dup('A1')}, NOR(NOR(A2, A2), NOR(A2, A2)))"
    _t2 = f"NOR(NOR(NOR(A1, A1), NOR(A1, A1)), NOR(A2, A2))"
    _xor2 = f"NOR(NOR({_t1}, {_t2}), NOR({_t1}, {_t2}))"
    _xnor2 = f"NOR({_xor2}, {_xor2})"

    if kind == "not":
        return _nor_dup(vs[0])
    if kind == "buf":
        return f"NOR({_nor_dup(_nor_dup('A1'))}, {_nor_dup(_nor_dup('A1'))})"
    if kind == "and":
        return ANDn(vs)
    if kind == "or":
        return ORn(vs)
    if kind == "nand":
        return NANDn(vs)
    if kind == "nor":
        return NORns(vs)
    if kind == "xor":
        return _xor2 if n == 2 else None
    if kind == "xnor":
        return _xnor2 if n == 2 else None
    return None


def build_canonical_nor_mapping(gate_type):
    """Decomposed gate lines (NOR-only) for get_mapped_circuit, or None if unsupported."""
    rhs = _canonical_nor_rhs(gate_type)
    if not rhs:
        return None
    return final_decompose_expression(rhs, "Y").rstrip("\n")


def get_circuit_from_response(response):
    """Parse LLM response to extract gate equations (from original code)."""
    text = _strip_code_fences(response) if response else ""
    LLM_circuit = ""
    lines = [ln for ln in text.split("\n") if ln.strip()]
    if not lines:
        return "Incorrect format"

    for line in lines:
        if line.strip().startswith("#"):
            continue
        if ("=" in line) and (not line.strip().startswith("wire ")):
            tmp_expression = line.split("=")[-1].strip().strip(";")
            if any(f"{g} " in tmp_expression for g in ["AND", "OR", "NAND", "NOR", "XOR", "XNOR"]):
                return "Incorrect format"
            tmp_op_net = line.split("=")[0].strip()
            decomposed = final_decompose_expression(tmp_expression, tmp_op_net)
            LLM_circuit += decomposed + "\n"

    if not LLM_circuit.strip():
        return "Incorrect format"
    return LLM_circuit.rstrip("\n")


def _fmt_sim_score(score):
    """Format SIM text-similarity for printing. ``0.0`` is valid — only ``None`` means unavailable."""
    return f'{score:.4f}' if score is not None else 'N/A'


SIM_THRESHOLD = 0.3

# One switch for slow LLM tasks: Task 1 (full netlist) and Task 2 (Boolean rewrite).
# True = skip those API calls for fast top-to-bottom runs; use Task 3 (canonical NOR) instead.
NOTEBOOK_SKIP_SLOW_LLM_TASKS = False  # Task 1 & 2 will call the LLM (slow). Set True for fast runs.

print('✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).')
print(f'SIM detection threshold: {SIM_THRESHOLD}')
print(f'NOTEBOOK_SKIP_SLOW_LLM_TASKS: {NOTEBOOK_SKIP_SLOW_LLM_TASKS}  (Task 1 & 2 LLM calls)')
random.seed(42)
print('Random seed set to 42 (used when REFERENCE_STRATEGY or task mapping uses random).')


✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).
SIM detection threshold: 0.3
NOTEBOOK_SKIP_SLOW_LLM_TASKS: False  (Task 1 & 2 LLM calls)
Random seed set to 42 (used when REFERENCE_STRATEGY or task mapping uses random).


In [3]:
#@title 0.3 Load Circuit & Setup OpenAI Client

import subprocess
import sys

# Load the test circuit
DESIGN = 'C432'
VARIANT = 'c432-CS320'

circuit_path = os.path.join(DATASET_PATH, DESIGN, VARIANT, 'topModule.v')
with open(circuit_path, 'r') as f:
    orig_file_contents = f.read()

gate_counts = characterize_generic_netlist(orig_file_contents)
total_gates = sum(gate_counts.values())

print(f'Loaded circuit: {DESIGN}/{VARIANT}')
print(f'Total gates: {total_gates}')
print(f'Gate types: {dict(gate_counts)}')
print(f'Circuit size: {len(orig_file_contents)} characters')

print(f'\nFirst 400 characters of circuit:')
print(orig_file_contents[:400] + '...')

def _pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=False)


try:
    from portkey_ai import Portkey
except ModuleNotFoundError:
    _pip_install("portkey-ai")
    from portkey_ai import Portkey

# Load HW6_LLMPirate/.env if present (cell 0.1 sets cwd to LLM_IP_assignment; walk parents).
try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    _pip_install("python-dotenv")
    from dotenv import load_dotenv

from pathlib import Path

_env_loaded = False
for _root in [Path.cwd(), *list(Path.cwd().parents)[:6]]:
    _env = _root / ".env"
    if _env.is_file():
        load_dotenv(_env)
        _env_loaded = True
        break
if _env_loaded:
    print("Loaded .env from:", str(_env))

OPENAI_BASE_URL = "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"
_k = (os.environ.get("OPENAI_API_KEY") or os.environ.get("NYU_GATEWAY_API_KEY") or "").strip()
OPENAI_API_KEY = _k if _k else None
# Use the course NYU AI Gateway key (OpenAI-style keys from openai.com will not work here).

LLM_CLIENT = None
if OPENAI_API_KEY:
    LLM_CLIENT = Portkey(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    print("\nPortkey API configured")
    print("Base URL:", OPENAI_BASE_URL)
    print(f"API key: loaded ({len(OPENAI_API_KEY)} chars)")
else:
    print("\nWARNING: No API key set. Set OPENAI_API_KEY or NYU_GATEWAY_API_KEY in the environment or HW6_LLMPirate/.env")


Loaded circuit: C432/c432-CS320
Total gates: 212
Gate types: {'nand_2': 64, 'not_1': 60, 'xor_2': 32, 'nor_2': 19, 'xnor_2': 18, 'nand_4': 14, 'and_9': 3, 'and_8': 1, 'nand_3': 1}
Circuit size: 10789 characters

First 400 characters of circuit:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
        keyIn_0_10,...
Loaded .env from: /Users/princydoshi/Desktop/LLM4ChipDesign-1/HW6_LLMPirate/.env

Portkey API configured
Base URL: https://ai-gateway.apps.cloud.rt.nyu.edu/v1
API key: loaded (28 chars)


## Task 1 — Direct Verilog rewriting

**Goal.** Ask the model to emit a **single** structural Verilog module that uses only the target primitives (here **NOR**), preserving ports and behavior.

**Why it is hard.** The netlist is large; models often truncate or emit invalid Verilog. There is no per-gate repair loop in this task.

**Speed.** With **`NOTEBOOK_SKIP_SLOW_LLM_TASKS = True`** (cell 0.2), the Task 1 code cell **returns immediately** — it does not build the large prompt or call the API. Set that flag to **False** (or **`TASK1_SKIP = False`** in the Task 1 code cell) to run the full direct rewrite.

**Evaluation.** Responses are cleaned for parsing, then checked with Yosys and the text-similarity path used in `evaluate_sim`.


In [4]:
#@title Task 1: Direct Verilog Rewriting Implementation

import time

# Configuration
# Full-chip direct rewrite (especially NOR on C432) = many LLM calls and often 15–60+ minutes.
# Default follows NOTEBOOK_SKIP_SLOW_LLM_TASKS (cell 0.2). Override here if needed:
TASK1_SKIP = globals().get("NOTEBOOK_SKIP_SLOW_LLM_TASKS", True)
# TASK1_SKIP = False  # uncomment to run Task 1 even when the notebook flag skips slow tasks
TASK1_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK1_MODEL = "@vertexai/gemini-2.5-pro"
# Larger max_tokens = fewer HTTP round-trips if your gateway honors it (often faster wall-clock).
TASK1_MAX_TOKENS = 32768  # NYU gateway: raise if you still hit length truncation every chunk.
TASK1_MAX_CONTINUATIONS = 24  # NOR+C432 may need many chunks; raise further if you still stop before endmodule.

task1_result = None
task1_sim_score = None
task1_evades = False
task1_functionally_correct = None

if TASK1_SKIP:
    print("\nTask 1: Direct Verilog Rewriting")
    print("⏭️ TASK1_SKIP=True — skipping direct full-netlist LLM call (fast; no prompt built).")
    print("   Set NOTEBOOK_SKIP_SLOW_LLM_TASKS=False in cell 0.2, or TASK1_SKIP=False in this cell, to run Task 1 (slow). For NOR+C432, prefer Task 3 (canonical mapping).")
elif LLM_CLIENT is None:
    print("\nTask 1: Direct Verilog Rewriting")
    print("⚠️ Skipping Task 1 execution (no Portkey client)")
else:
    task1_prompt = f"""
You refactor gate-level Verilog for readability and a target gate library.

Target primitives (use only these gate types for combinational logic): {TASK1_TARGET_GATES}

Hard requirements:
- Preserve exact input/output behavior vs the original netlist (same truth table).
- The rewritten module MUST be named `top` with the SAME ports in the SAME order as the original.
- Output ONE complete structural Verilog-2001 module from `module` through `endmodule`.
- Your reply MUST be ONLY Verilog source code: no markdown, no ``` fences, no explanation, no bullet lists.
- Do not stop early: the netlist is large — finish through `endmodule` with all gates and wires.

Original netlist:
{orig_file_contents}
"""

    print("Task 1 Prompt Summary:")
    print("=" * 50)
    print(f"Target gates: {TASK1_TARGET_GATES}")
    print(f"Model: {TASK1_MODEL}")
    print(f"Prompt length: {len(task1_prompt)} characters")
    print("\nFirst 400 chars of prompt:")
    print(task1_prompt[:400] + "...")

    print("\n" + "=" * 50)
    print("EXECUTING TASK 1 (this can take a long time — each API round may be 1–10+ min)...")
    print("=" * 50)

    def _task1_continuation_message(partial: str) -> str:
        """Give the model an exact tail anchor so it continues instead of restarting or explaining."""
        partial = partial or ""
        tail = partial[-12000:] if len(partial) > 12000 else partial
        return (
            "Your previous answer was truncated before `endmodule`. "
            "Continue the SAME module `top` immediately after the text in the block below.\n"
            "Rules: output ONLY additional Verilog lines (no markdown fences, no commentary). "
            "Do not repeat any full line that already appears above the block; append new lines only. "
            "End with a line that is exactly: endmodule\n\n"
            "---LAST_PART_OF_YOUR_OUTPUT---\n"
            f"{tail}\n"
            "---CONTINUE_BELOW---"
        )

    try:
        _t0 = time.monotonic()
        response = LLM_CLIENT.chat.completions.create(
            model=TASK1_MODEL,
            messages=[{"role": "user", "content": task1_prompt}],
            temperature=0.3,
            max_tokens=TASK1_MAX_TOKENS,
        )
        print(f"   ⏱ first response: {time.monotonic() - _t0:.1f}s")
        choice0 = response.choices[0]
        task1_result = choice0.message.content or ""
        fr = getattr(choice0, "finish_reason", None)
        _cont_max = TASK1_MAX_CONTINUATIONS
        _cont_try = 0
        while _cont_try < _cont_max and _needs_verilog_continuation(task1_result):
            _prev_len = len(task1_result)
            _cont_try += 1
            print(f"⚠️ Response missing endmodule — continuation request {_cont_try}/{_cont_max}...")
            if fr == "length":
                print("   (previous chunk ended at max_tokens; requesting more output)")
            _t1 = time.monotonic()
            cont = LLM_CLIENT.chat.completions.create(
                model=TASK1_MODEL,
                messages=[
                    {"role": "user", "content": task1_prompt},
                    {"role": "assistant", "content": task1_result},
                    {"role": "user", "content": _task1_continuation_message(task1_result)},
                ],
                temperature=0.2,
                max_tokens=TASK1_MAX_TOKENS,
            )
            chunk = cont.choices[0].message.content or ""
            fr = getattr(cont.choices[0], "finish_reason", None)
            task1_result = (task1_result.rstrip() + "\n" + chunk).strip()
            print(
                f"   → cumulative output: {len(task1_result)} chars "
                f"(+{len(task1_result) - _prev_len} from this chunk) "
                f"[chunk API time {time.monotonic() - _t1:.1f}s]"
            )
            if len(chunk) < 16 and not _needs_verilog_continuation(task1_result):
                break
            if len(task1_result) - _prev_len < 8 and _cont_try >= 3:
                print(
                    "⚠️ Continuation added almost no text — gateway/model may be stuck. "
                    "Try raising TASK1_MAX_TOKENS if supported, or use NAND / AND+NOT (smaller NOR blow-up)."
                )

        if _needs_verilog_continuation(task1_result):
            print(
                f"❌ Still no `endmodule` after {_cont_max} continuation(s) "
                f"({len(task1_result)} chars). NOR-only mappings can exceed practical output limits; "
                "raise TASK1_MAX_TOKENS / TASK1_MAX_CONTINUATIONS or try a less verbose gate set."
            )

        print(f"✅ LLM Response received ({len(task1_result)} chars)")
        print("First 400 chars of response:")
        print(task1_result[:400] + "...")

        task1_incomplete = _needs_verilog_continuation(task1_result)
        if task1_incomplete:
            print("\n⚠️ Skipping Yosys and SIM (incomplete module — no `endmodule`).")
            task1_functionally_correct = None
            task1_sim_score = None
            task1_evades = False
            overall_success = False
            print(f"\n📊 TASK 1 RESULTS:")
            print("Functional Correctness: ⚠️ UNKNOWN (incomplete Verilog)")
            print("SIM Score: N/A")
            print(f"Model Used: {TASK1_MODEL}")
            print(f"Target Gates: {TASK1_TARGET_GATES}")
            print("Overall Success: ❌ NO")
        else:
            # Evaluate functional correctness
            print("\n🔍 Checking functional equivalence...")
            equiv_result, equiv_details = check_functional_equivalence(
                orig_file_contents, task1_result, "task1"
            )
            task1_functionally_correct = equiv_result
            print(f"Functional check: {equiv_details}")

            # Evaluate with SIM
            print("\n🔍 Evaluating with SIM detector...")
            task1_sim_score = evaluate_sim(orig_file_contents, task1_result, "task1")
            task1_evades = task1_sim_score is not None and task1_sim_score < SIM_THRESHOLD

            print(f"\n📊 TASK 1 RESULTS:")
            print(f"Functional Correctness: {'✅ YES' if task1_functionally_correct else '❌ NO' if task1_functionally_correct is False else '⚠️ UNKNOWN'}")
            print(f"SIM Score: {_fmt_sim_score(task1_sim_score)}")
            print(f"Evades Detection: {'✅ YES' if task1_evades else '❌ NO'}")
            print(f"Model Used: {TASK1_MODEL}")
            print(f"Target Gates: {TASK1_TARGET_GATES}")

            # Overall success requires BOTH functional correctness AND SIM evasion
            overall_success = task1_functionally_correct and task1_evades
            print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except KeyboardInterrupt:
        _n = len(task1_result or "")
        print(
            "\n⚠️ Task 1 interrupted during an API call (kernel Stop or Ctrl+C). "
            "That is normal if you stopped a slow request. "
            f"Partial Verilog may still be in `task1_result` (~{_n} chars)—copy it if you need it. "
            "Continuations can take minutes for NOR-only C432; rerun the cell when ready, or use Task 3 / a smaller gate library."
        )
        raise
    except Exception as e:
        print(f"❌ Task 1 failed: {e}")
        task1_result = None
        task1_sim_score = None
        task1_evades = False
        task1_functionally_correct = None


Task 1 Prompt Summary:
Target gates: NOR
Model: @vertexai/gemini-2.5-pro
Prompt length: 11455 characters

First 400 chars of prompt:

You refactor gate-level Verilog for readability and a target gate library.

Target primitives (use only these gate types for combinational logic): NOR

Hard requirements:
- Preserve exact input/output behavior vs the original netlist (same truth table).
- The rewritten module MUST be named `top` with the SAME ports in the SAME order as the original.
- Output ONE complete structural Verilog-2001 m...

EXECUTING TASK 1 (this can take a long time — each API round may be 1–10+ min)...
   ⏱ first response: 236.0s
⚠️ Response missing endmodule — continuation request 1/24...
   → cumulative output: 3401 chars (+1604 from this chunk) [chunk API time 239.8s]
⚠️ Response missing endmodule — continuation request 2/24...
   → cumulative output: 10108 chars (+6707 from this chunk) [chunk API time 257.9s]
⚠️ Response missing endmodule — continuation request 3/24...
  

### Task 1 — Results and discussion

| Quantity | Value (last recorded run) |
|----------|-------|
| Model | `@vertexai/gemini-2.5-pro` |
| Target library | NOR-only |
| Continuations used | **5 / 24** (still no complete `module`…`endmodule` in extracted Verilog) |
| Approx. response size | **~40k** characters |
| Text similarity proxy (difflib, `orig_task1.v` vs `pirated_task1.v`) | **0.2185** |
| Below SIM threshold 0.3? | **Yes** |
| Yosys equivalence | **Unknown** — checker reports **incomplete** transformed Verilog (no `endmodule` after cleaning), so **no miter** |
| Overall success | **No** |

**Discussion.** Direct rewrite can **evade** SIM yet still be **unfinishable** at this scale (markdown fences, truncation, or structure that never closes). Until Yosys sees a complete module, **functional correctness is undecided** — not the same as “equivalent.”


## Task 2 — Boolean intermediate format

**Goal.** Convert the netlist to Boolean equations, prompt the model to rewrite them using only **NOR**, then convert back to Verilog.

**Rationale.** The intermediate form is closer to algebraic reasoning than raw gate instances.

**Speed.** When **`NOTEBOOK_SKIP_SLOW_LLM_TASKS = True`** (cell 0.2), the Task 2 code cell **skips before** Boolean conversion and prompt build. Override with **`TASK2_SKIP = False`** in that cell if needed.

**Evaluation.** Same pipeline: Yosys equivalence + text similarity. The Task 2 code cell uses **`@vertexai/gemini-2.5-pro`** (NYU gateway), aligned with Tasks 1 and 3. It respects **`NOTEBOOK_SKIP_SLOW_LLM_TASKS`** from cell 0.2 unless you override **`TASK2_SKIP`**.


In [5]:
#@title Task 2: Boolean Format Rewriting Implementation

TASK2_SKIP = globals().get("NOTEBOOK_SKIP_SLOW_LLM_TASKS", True)
# TASK2_SKIP = False  # uncomment to run Task 2 even when NOTEBOOK_SKIP_SLOW_LLM_TASKS is True
TASK2_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK2_MODEL = "@vertexai/gemini-2.5-pro"  # use same gateway model as Task 1 / 3 (NYU Portkey)

task2_result = None
task2_sim_score = None
task2_evades = False
task2_functionally_correct = None

if TASK2_SKIP:
    print("\nTask 2: Boolean Format Rewriting")
    print("⏭️ TASK2_SKIP=True — skipping Boolean conversion + LLM (fast; no prompt built).")
    print("   Set NOTEBOOK_SKIP_SLOW_LLM_TASKS=False in cell 0.2, or TASK2_SKIP=False above, to run Task 2.")
elif LLM_CLIENT is None:
    print("\nTask 2: Boolean Format Rewriting")
    print("⚠️ Skipping Task 2 execution (no Portkey client)")
else:
    circ_in_Boolean_format, remaining_orig_lines = convert_circuit_to_Boolean_format(orig_file_contents)

    print("Boolean Format Conversion:")
    print("=" * 30)
    print(f"Boolean equations: {len([l for l in circ_in_Boolean_format.splitlines() if l.strip()])} lines")
    print(f"Remaining structural lines: {len(remaining_orig_lines.splitlines())} lines")
    print("\nFirst 300 chars of Boolean format:")
    print(circ_in_Boolean_format[:300] + "...")

    task2_prompt = f"""
Rewrite these Boolean equations using ONLY {TASK2_TARGET_GATES} as the combinational operator.

Strict requirements:
1. Preserve exact functionality (same truth table for every named signal).
2. Keep the same primary input/output net names; you may add intermediate wires (e.g. t0, t1).
3. One equation per line:  signal = {TASK2_TARGET_GATES}( ... )  with parentheses and commas.
4. For NOR-only: remember NOT(x) is NOR(x,x); build AND/OR/NAND/XOR from NOR only.
5. Output ONLY equations — no markdown fences, no commentary.

Equations:
{circ_in_Boolean_format}

Rewritten equations:
"""

    print(f"\nTask 2 Configuration:")
    print(f"Target gates: {TASK2_TARGET_GATES}")
    print(f"Model: {TASK2_MODEL}")
    print(f"Prompt length: {len(task2_prompt)} characters")

    print("\n" + "=" * 50)
    print("EXECUTING TASK 2...")
    print("=" * 50)

    try:
        response = LLM_CLIENT.chat.completions.create(
            model=TASK2_MODEL,
            messages=[{"role": "user", "content": task2_prompt}],
            temperature=0.3,
            max_tokens=16384,
        )
        llm_boolean_response = response.choices[0].message.content or ""
        llm_boolean_response = _strip_code_fences(llm_boolean_response)

        print(f"✅ LLM Response received ({len(llm_boolean_response)} chars)")
        print("First 300 chars of Boolean response:")
        print(llm_boolean_response[:300] + "...")

        # Convert back to Verilog format
        print("\n🔄 Converting back to Verilog format...")
        task2_result = convert_circuit_from_Boolean_format(llm_boolean_response, remaining_orig_lines)

        print(f"✅ Converted back to Verilog ({len(task2_result)} chars)")

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task2_result, "task2"
        )
        task2_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task2_sim_score = evaluate_sim(orig_file_contents, task2_result, "task2")
        task2_evades = task2_sim_score is not None and task2_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 2 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task2_functionally_correct else '❌ NO' if task2_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {_fmt_sim_score(task2_sim_score)}")
        print(f"Evades Detection: {'✅ YES' if task2_evades else '❌ NO'}")
        print(f"Model Used: {TASK2_MODEL}")
        print(f"Target Gates: {TASK2_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task2_functionally_correct and task2_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except KeyboardInterrupt:
        print("\n⚠️ Task 2 interrupted (e.g. Stop during API call). Rerun the cell to retry.")
        raise
    except Exception as e:
        print(f"❌ Task 2 failed: {e}")
        task2_result = None
        task2_sim_score = None
        task2_evades = False
        task2_functionally_correct = None


Boolean Format Conversion:
Boolean equations: 212 lines
Remaining structural lines: 13 lines

First 300 chars of Boolean format:
KeyWire_0[0] = NOT(N1)
N118 = XNOR(keyIn_0_0, KeyWire_0[0])
N119 = NOT(N4)
KeyWire_0[1] = NOT(N11)
N122 = XNOR(keyIn_0_1, KeyWire_0[1])
N123 = NOT(N17)
KeyWire_0[2] = NOT(N24)
KeyNOTWire_0[0] = XNOR(keyIn_0_2, KeyWire_0[2])
N126 = NOT(KeyNOTWire_0[0])
N127 = NOT(N30)
KeyWire_0[3] = NOT(N37)
N130 = X...

Task 2 Configuration:
Target gates: NOR
Model: @vertexai/gemini-2.5-pro
Prompt length: 6598 characters

EXECUTING TASK 2...
✅ LLM Response received (1023 chars)
First 300 chars of Boolean response:
KeyWire_0[0] = NOR(N1, N1)
t0 = NOR(keyIn_0_0, keyIn_0_0)
t1 = NOR(t0, KeyWire_0[0])
t2 = NOR(KeyWire_0[0], KeyWire_0[0])
t3 = NOR(keyIn_0_0, t2)
N118 = NOR(t1, t3)
N119 = NOR(N4, N4)
KeyWire_0[1] = NOR(N11, N11)
t4 = NOR(keyIn_0_1, keyIn_0_1)
t5 = NOR(t4, KeyWire_0[1])
t6 = NOR(KeyWire_0[1], KeyWir...

🔄 Converting back to Verilog format...
✅ Converted back to Veri

### Task 2 — Results and discussion

| Quantity | Value (last recorded run) |
|----------|-------|
| Model (configured) | `@vertexai/gemini-2.5-pro` |
| Target library | NOR-only |
| Text similarity proxy (difflib, `orig_task2.v` vs `pirated_task2.v`) | **0.1250** |
| Below SIM threshold 0.3? | **Yes** |
| Yosys equivalence (`truth_task2.v` vs `gen_task2.v`) | **Not equivalent** — SAT counterexample |
| Overall success | **No** |

**Discussion.** The Boolean rewrite is easier to generate than monolithic Verilog, but **algebraic mistakes** still break functionality. Evasion without equivalence is not a viable “piracy” outcome under formal scrutiny.


## Task 3 — Divide and conquer (no feedback)

**Goal.** For each **gate type** (`nand_2`, `xor_2`, …), obtain a NOR-only decomposition, then substitute into the full netlist.

**Canonical NOR mode.** With `USE_CANONICAL_NOR_MAPPINGS = True`, the notebook uses deterministic NOR-universal templates instead of per-type LLM guesses. This is the most reliable path in this assignment.

**Evaluation.** Full-netlist equivalence in Yosys plus text similarity on the mapped Verilog.


In [6]:
#@title Task 3: Divide & Conquer (No Feedback) Implementation

# Characterize the circuit to get all gate types and query templates
# (This is the "divide" step: one LLM query per gate type)
gate_counts = characterize_generic_netlist(orig_file_contents)
query_templates = create_LLM_circuit_query_template(gate_counts)

# Configuration
TASK3_TARGET_GATES = ["NOR"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK3_MODEL = "@vertexai/gemini-2.5-pro"
# When True and target is NOR-only, use math-correct NOR-universal mappings (recommended).
USE_CANONICAL_NOR_MAPPINGS = True
TASK3_USE_CANONICAL = (
    USE_CANONICAL_NOR_MAPPINGS
    and len(TASK3_TARGET_GATES) == 1
    and str(TASK3_TARGET_GATES[0]).upper() == "NOR"
)
# Gates skipped by design: with canonical NOR, map NOT→NOR too; only BUF stays skipped if present.
SKIP_GATE_TYPES = ['buf_1'] if TASK3_USE_CANONICAL else ['not_1', 'buf_1']
non_trivial_gates = [g for g in gate_counts if g not in SKIP_GATE_TYPES]
skipped_gates    = [g for g in gate_counts if g in SKIP_GATE_TYPES]

print("Gate types found:", list(gate_counts.keys()))
print(f"Non-trivial gate types to map: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial, no rephrasing needed): {skipped_gates}")
if TASK3_USE_CANONICAL:
    print("Using canonical NOR-universal mappings (no LLM per gate type).")

# Prompt template (from original GPT_communication_script.py)
def create_task3_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_rule = f"Use only the {target_gates[0]} Boolean operator."
    else:
        gate_rule = f"Use only {' and/or '.join(target_gates)} Boolean operators."
    return (
        "Can you refactor this Verilog circuit following these instructions? "
        f"1) {gate_rule} "
        "2) Ensure that the final functionality remains the same. "
        "3) Just give me the new circuit and nothing else. "
        "4) Generate your response in the following format: <output> = <gate type> (<inputs>).\n"
        + template
    )

print(f"\nTask 3 Configuration:")
print(f"Target gates: {TASK3_TARGET_GATES}")
print(f"Model: {TASK3_MODEL}")

# Initialize results
task3_circuit_mapping = {}
task3_final_circuit = None
task3_sim_score = None
task3_evades = False
task3_functionally_correct = None

# Task 3 Execution (canonical NOR mode runs without an API client)
if LLM_CLIENT is not None or TASK3_USE_CANONICAL:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 3...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK3_TARGET_GATES)

        # Initialize all gate types with empty mappings (unmapped gates stay as-is)
        for gate_type in gate_counts:
            task3_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, ask the LLM for a single-shot rephrasing
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            if TASK3_USE_CANONICAL:
                cm = build_canonical_nor_mapping(gate_type)
                if cm:
                    print(f"  ✅ Canonical NOR mapping (verified template)")
                    task3_circuit_mapping[gate_type][target_key] = [cm]
                    continue
                print(f"  ⚠️ No canonical template — trying LLM")
            if LLM_CLIENT is None:
                print(f"  ⚠️ No API client — skipping LLM for {gate_type}")
                continue
            prompt = create_task3_prompt(template, TASK3_TARGET_GATES)

            response = LLM_CLIENT.chat.completions.create(
                model=TASK3_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )

            llm_response = response.choices[0].message.content or ""
            LLM_circuit = get_circuit_from_response(llm_response)

            if "Incorrect format" in LLM_circuit:
                print(f"  ⚠️ Could not parse LLM response — keeping original gate")
                continue

            print(f"  ✅ Mapping: {LLM_circuit.strip()}")
            task3_circuit_mapping[gate_type][target_key] = [LLM_circuit]

        mapped_count = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")

        # Apply all mappings to the full circuit ("conquer" step)
        print("\n🔄 Applying gate mappings to full circuit...")
        task3_final_circuit = get_mapped_circuit(orig_file_contents, task3_circuit_mapping, target_key)

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task3_final_circuit, "task3"
        )
        task3_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task3_sim_score = evaluate_sim(orig_file_contents, task3_final_circuit, "task3")
        task3_evades = task3_sim_score is not None and task3_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 3 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Functional Correctness: {'✅ YES' if task3_functionally_correct else '❌ NO' if task3_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {_fmt_sim_score(task3_sim_score)}")
        print(f"Evades Detection: {'✅ YES' if task3_evades else '❌ NO'}")
        print(f"Model Used: {TASK3_MODEL}")
        print(f"Target Gates: {TASK3_TARGET_GATES}")
        overall_success = task3_functionally_correct and task3_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")
        # Snapshot for Task 5 summary (Task 4 overwrites `non_trivial_gates` later).
        TASK3_SUMMARY_MAPPED = mapped_count
        TASK3_SUMMARY_NONTRIVIAL = len(non_trivial_gates)
        TASK3_NONTRIVIAL_GATES = tuple(non_trivial_gates)

    except KeyboardInterrupt:
        print("\n⚠️ Task 3 interrupted. Partial mappings may remain in `task3_circuit_mapping`.")
        raise
    except Exception as e:
        print(f"❌ Task 3 failed: {e}")
        task3_circuit_mapping = {}
        task3_final_circuit = None
        task3_sim_score = None
        task3_evades = False
        task3_functionally_correct = None
else:
    print("⚠️ Skipping Task 3 (set API key or enable TASK3_USE_CANONICAL with NOR target)")


Gate types found: ['nand_2', 'not_1', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Non-trivial gate types to map: ['nand_2', 'not_1', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Using canonical NOR-universal mappings (no LLM per gate type).

Task 3 Configuration:
Target gates: ['NOR']
Model: @vertexai/gemini-2.5-pro

EXECUTING TASK 3...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  ✅ Canonical NOR mapping (verified template)

--- Gate type: not_1  (template: Y = NOT(A1)) ---
  ✅ Canonical NOR mapping (verified template)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  ✅ Canonical NOR mapping (verified template)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  ✅ Canonical NOR mapping (verified template)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  ✅ Canonical NOR mapping (verified template)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  ✅ Canonical NOR mapping (verified template)

-

### Task 3 — Results and discussion

| Quantity | Value (last recorded run) |
|----------|-------|
| Model label | `@vertexai/gemini-2.5-pro` (mapping via **canonical NOR** templates) |
| Target library | NOR-only |
| Gate types mapped | **9 / 9** non-trivial |
| Text similarity proxy (difflib, `orig_task3.v` vs `pirated_task3.v`) | **0.0147** |
| Below SIM threshold 0.3? | **Yes** |
| Yosys equivalence (`truth_task3.v` vs `gen_task3.v`) | **Equivalent** (`no model found: SUCCESS`) |
| Overall success | **Yes** |

**Discussion.** Deterministic NOR decomposition is the standout: it minimizes LLM error while still drastically changing the netlist text.


## Task 4 — Divide and conquer (with feedback)

**Goal.** Same as Task 3, but with retries and user-provided feedback messages when parsing fails.

**Configuration in this submission.** `USE_CANONICAL_NOR_MAPPINGS = False` so mappings come from the **LLM** and the feedback machinery is meaningful. (When canonical mode is on, Task 4 collapses to the template shortcut.)

**Evaluation.** Identical metrics to Task 3.


In [7]:
#@title Task 4: Divide & Conquer WITH Iterative Feedback Implementation

# Reuse gate characterization from Task 3 when available; otherwise derive from the loaded netlist.
if 'gate_counts' not in globals():
    gate_counts = characterize_generic_netlist(orig_file_contents)
if 'query_templates' not in globals():
    query_templates = create_LLM_circuit_query_template(gate_counts)

# Configuration
TASK4_TARGET_GATES = ["NOR"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK4_MODEL = "@vertexai/gemini-2.5-pro"  # keep constant for Task 3 vs Task 4 comparison
TASK4_MAX_TRIALS = 5  # Max feedback rounds per gate type (matches original source)
USE_CANONICAL_NOR_MAPPINGS = False  # Required run: disable canonical mapping to test feedback loop
TASK4_USE_CANONICAL = (
    USE_CANONICAL_NOR_MAPPINGS
    and len(TASK4_TARGET_GATES) == 1
    and str(TASK4_TARGET_GATES[0]).upper() == "NOR"
)
SKIP_GATE_TYPES = ["buf_1"] if TASK4_USE_CANONICAL else ["not_1", "buf_1"]
non_trivial_gates = [g for g in gate_counts if g not in SKIP_GATE_TYPES]
skipped_gates = [g for g in gate_counts if g in SKIP_GATE_TYPES]
if TASK4_USE_CANONICAL:
    print("Task 4: canonical NOR mode — feedback loop skipped (mappings are exact).")

print(f"Gate types to process: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial): {skipped_gates}")

# Prompt template (from original GPT_communication_script.py)
def create_task4_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_rule = f"Use only the {target_gates[0]} Boolean operator."
    else:
        gate_rule = f"Use only {' and/or '.join(target_gates)} Boolean operators."
    return (
        "Can you refactor this Verilog circuit following these instructions? "
        f"1) {gate_rule} "
        "2) Ensure that the final functionality remains the same. "
        "3) Just give me the new circuit and nothing else. "
        "4) Generate your response in the following format: <output> = <gate type> (<inputs>).\n"
        + template
    )

def create_task4_format_feedback(template, target_gates):
    return (
        "This is not the correct format. Can you try again in this format: "
        "<output> = <gate type> (<inputs>)? "
        f"Use only {', '.join(target_gates)} operator(s). "
        "Below is the original circuit:\n" + template
    )

def create_task4_functional_feedback(template, target_gates):
    return (
        "This is not correct because the functionality is not the same as the original circuit. "
        "Can you try again? Below is the original circuit:\n" + template
    )

print(f"\nTask 4 Configuration:")
print(f"Target gates: {TASK4_TARGET_GATES}")
print(f"Model: {TASK4_MODEL}")
print(f"Max trials per gate type: {TASK4_MAX_TRIALS}")

# Initialize results
task4_circuit_mapping = {}
task4_final_circuit = None
task4_sim_score = None
task4_evades = False
task4_functionally_correct = None
task4_gate_trial_counts = {}

# Task 4 Execution (canonical NOR mode runs without an API client)
if LLM_CLIENT is not None or TASK4_USE_CANONICAL:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 4...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK4_TARGET_GATES)

        # Initialize all gate types with empty mappings
        for gate_type in gate_counts:
            task4_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, query LLM with iterative feedback on failure
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            if TASK4_USE_CANONICAL:
                cm = build_canonical_nor_mapping(gate_type)
                if cm:
                    print(f"  ✅ Canonical NOR mapping (no LLM trials)")
                    task4_circuit_mapping[gate_type][target_key] = [cm]
                    task4_gate_trial_counts[gate_type] = 1
                    continue
            if LLM_CLIENT is None:
                print(f"  ⚠️ No API client — skipping LLM for {gate_type}")
                continue
            messages = [{"role": "user", "content": create_task4_prompt(template, TASK4_TARGET_GATES)}]
            num_trials = 0
            success = False

            while num_trials < TASK4_MAX_TRIALS:
                num_trials += 1
                print(f"  Trial {num_trials}/{TASK4_MAX_TRIALS}")

                # Use full conversation history for context (feedback loop)
                response = LLM_CLIENT.chat.completions.create(
                    model=TASK4_MODEL,
                    messages=messages,
                    temperature=0.7,
                    max_tokens=500
                )
                llm_response = response.choices[0].message.content or ""
                messages.append({"role": "assistant", "content": llm_response})

                LLM_circuit = get_circuit_from_response(llm_response)

                if "Incorrect format" in LLM_circuit:
                    print(f"  ⚠️ Format error — sending feedback")
                    feedback = create_task4_format_feedback(template, TASK4_TARGET_GATES)
                    messages.append({"role": "user", "content": feedback})
                    continue

                print(f"  ✅ Mapping: {LLM_circuit.strip()}")
                task4_circuit_mapping[gate_type][target_key] = [LLM_circuit]
                success = True
                break

            task4_gate_trial_counts[gate_type] = num_trials
            if not success:
                print(f"  ❌ No valid mapping after {TASK4_MAX_TRIALS} trials — keeping original gate")

        mapped_count = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
        avg_trials = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")

        # Apply all mappings to the full circuit
        print("\n🔄 Applying gate mappings to full circuit...")
        task4_final_circuit = get_mapped_circuit(orig_file_contents, task4_circuit_mapping, target_key)

        # Check functional equivalence on the full mapped circuit
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task4_final_circuit, "task4"
        )
        task4_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task4_sim_score = evaluate_sim(orig_file_contents, task4_final_circuit, "task4")
        task4_evades = task4_sim_score is not None and task4_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 4 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")
        print(f"Functional Correctness: {'✅ YES' if task4_functionally_correct else '❌ NO' if task4_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {_fmt_sim_score(task4_sim_score)}")
        print(f"Evades Detection: {'✅ YES' if task4_evades else '❌ NO'}")
        print(f"Model Used: {TASK4_MODEL}")
        print(f"Target Gates: {TASK4_TARGET_GATES}")
        overall_success = task4_functionally_correct and task4_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")
        TASK4_SUMMARY_MAPPED = mapped_count
        TASK4_SUMMARY_NONTRIVIAL = len(non_trivial_gates)
        TASK4_SUMMARY_AVG_TRIALS = avg_trials
        TASK4_NONTRIVIAL_GATES = tuple(non_trivial_gates)

        # Show per-gate-type trial counts
        print("\nPer-gate-type trial counts:")
        for gt in gate_counts:
            if gt in SKIP_GATE_TYPES:
                print(f"  {gt}: — (trivial gate, skipped by design)")
            elif gt in task4_gate_trial_counts:
                trials = task4_gate_trial_counts[gt]
                mapped = '✅' if task4_circuit_mapping.get(gt) else '❌'
                print(f"  {gt}: {trials} trial(s) {mapped}")
            else:
                print(f"  {gt}: — (not processed — no API or skipped)")

    except KeyboardInterrupt:
        print("\n⚠️ Task 4 interrupted. Partial mappings may remain in `task4_circuit_mapping`.")
        raise
    except Exception as e:
        print(f"❌ Task 4 failed: {e}")
        task4_circuit_mapping = {}
        task4_final_circuit = None
        task4_sim_score = None
        task4_evades = False
        task4_functionally_correct = None
        task4_gate_trial_counts = {}
else:
    print("⚠️ Skipping Task 4 (set API key or enable TASK4_USE_CANONICAL with NOR target)")


Gate types to process: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial): ['not_1']

Task 4 Configuration:
Target gates: ['NOR']
Model: @vertexai/gemini-2.5-pro
Max trials per gate type: 5

EXECUTING TASK 4...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  Trial 1/5
  ⚠️ Format error — sending feedback
  Trial 2/5
  ⚠️ Format error — sending feedback
  Trial 3/5
  ⚠️ Format error — sending feedback
  Trial 4/5
  ✅ Mapping: W1 = NOR(A1, A1)
W2 = NOR(A)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOR(A1, A1)
N2 = NOR(A)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: Y = NOR(A1, A2)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOR(A1, A1)
N2 = NOR()

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  Trial 1/5
  ⚠️ Format error — sending feedback
  Trial 2/5
  ⚠️ Format error — sending feedback
  Trial 3

### Task 4 — Results and discussion

| Quantity | Value (last recorded run) |
|----------|-------|
| Model | `@vertexai/gemini-2.5-pro` |
| Target library | NOR-only |
| Canonical NOR | **Disabled** (`USE_CANONICAL_NOR_MAPPINGS = False`) — LLM-produced mappings |
| Gate types with a stored mapping | **7 / 8** non-trivial (`nand_4` often failed format after 5 trials) |
| Avg trials / gate type | **~2.1** |
| Text similarity proxy (difflib, `orig_task4.v` vs `pirated_task4.v`) | **0.2847** |
| Below SIM threshold 0.3? | **Yes** |
| Yosys equivalence (`truth_task4.v` vs `gen_task4.v`) | **Not equivalent** — SAT counterexample |
| Overall success | **No** |

**Discussion.** Feedback fixes **format** sometimes but not **semantics** — accepted lines like `NOR(A)` or `NOR()` are not valid universal NOR decompositions. Mixed good/bad per-type mappings **compose** to a wrong full circuit while SIM can still look “stealthy.”


## Task 5 — Reference baseline and comparison

**Goal.** Compare Tasks 1–4 against **cached** gate mappings from the original LLMPirate research artifacts (`cached_circuit_mapping_*.pkl`).

**Caveat.** Cache coverage may skip rare wide gates; partial mapping can break equivalence even when SIM improves.

**What the code cell prints.** A short **“how to read Yosys vs SIM”** prologue (semantic correctness vs textual difference), the **summary table**, two **rankings** (lab definition vs **correctness-first** for reports), and **insights** that treat lowest SIM among **Yosys-verified** runs separately from the global minimum SIM.


In [8]:
#@title Task 5: Reference Baseline + Strategy Comparison

from collections import defaultdict
import sys

if 'orig_file_contents' not in globals():
    raise RuntimeError('Run cell 0.3 (load circuit) before Task 5.')
if 'gate_counts' not in globals():
    gate_counts = characterize_generic_netlist(orig_file_contents)

# ─────────────────────────────────────────────────────────────
# PART A: Reference Baseline — Original Research Cached Mappings
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("PART A: REFERENCE BASELINE")
print("=" * 60)

_ok_self, _msg_self = check_functional_equivalence(orig_file_contents, orig_file_contents, 'sanity_self')
print(f"Sanity check (gold vs same netlist): {_msg_self}")

# Reference cache selector
REFERENCE_LLM = 'GPT3dot5'   # Try: 'GPT3dot5', 'GPT4', 'Claude', 'Gemini', 'llama3'
# 'deterministic' = same mapping choice every run (sorted key order). Reproducible.
# 'random' = different mix each run — expect variance in SIM and sometimes in formal equiv.
REFERENCE_STRATEGY = 'deterministic'

mapping_path = os.path.join(DATA_DIR, 'src', f'cached_circuit_mapping_{REFERENCE_LLM}.pkl')
try:
    with open(mapping_path, 'rb') as f:
        reference_mapping = pickle.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Missing cache file: {mapping_path}. Run cell 0.1 so LLM_IP_assignment is cloned, or set REFERENCE_LLM to a file that exists under src/."
    ) from None

print(f"Loaded cached mapping: {REFERENCE_LLM}")
print(f"\nCoverage for this circuit's gate types:")
for gt in gate_counts:
    strategies = reference_mapping.get(gt, {})
    if strategies:
        print(f"  ✅ {gt}: {list(strategies.keys())}")
    else:
        print(f"  ⬜ {gt}: no cached mapping (gate kept as-is)")

print(f"\n🔄 Applying {REFERENCE_LLM} cached mappings (strategy: {REFERENCE_STRATEGY})...")
ref_circuit = get_mapped_circuit(orig_file_contents, reference_mapping, REFERENCE_STRATEGY)

print("\n🔍 Checking functional equivalence...")
ref_equiv, ref_details = check_functional_equivalence(orig_file_contents, ref_circuit, 'reference')
print(f"Functional check: {ref_details}")

print("\n🔍 Evaluating with SIM detector...")
ref_sim_score = evaluate_sim(orig_file_contents, ref_circuit, 'reference')
ref_evades = ref_sim_score is not None and ref_sim_score < SIM_THRESHOLD

print(f"\n📊 REFERENCE BASELINE RESULTS ({REFERENCE_LLM}):")
print(f"Functional Correctness: {'✅ YES' if ref_equiv else '❌ NO' if ref_equiv is False else '⚠️ UNKNOWN'}")
print(f"SIM Score: {_fmt_sim_score(ref_sim_score)}")
print(f"Evades Detection: {'✅ YES' if ref_evades else '❌ NO'}")

# ─────────────────────────────────────────────────────────────
# PART B: Strategy Comparison
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART B: COMPREHENSIVE STRATEGY COMPARISON")
print("=" * 60)

all_results = []

_g = globals()

# Task 1 Results (include row whenever Task 1 cell produced output; SIM may be N/A)
if _g.get("task1_result") is not None and _g.get("TASK1_MODEL") is not None:
    all_results.append({
        'Task': 'Task 1: Direct Verilog',
        'Method': 'Direct LLM rewriting',
        'SIM_Score': task1_sim_score,
        'Functional': task1_functionally_correct,
        'Overall_Success': bool(task1_functionally_correct and task1_evades),
        'Model': TASK1_MODEL,
        'Notes': 'Single-shot, full circuit'
    })

# Task 2 Results
if _g.get("task2_result") is not None and _g.get("TASK2_MODEL") is not None:
    all_results.append({
        'Task': 'Task 2: Boolean Format',
        'Method': 'Boolean intermediate format',
        'SIM_Score': task2_sim_score,
        'Functional': task2_functionally_correct,
        'Overall_Success': bool(task2_functionally_correct and task2_evades),
        'Model': TASK2_MODEL,
        'Notes': 'Via Boolean equations'
    })

# Task 3 Results — prefer TASK3_NONTRIVIAL_GATES + task3_circuit_mapping (robust after Task 4 runs)
if _g.get("task3_final_circuit") is not None and _g.get("TASK3_MODEL") is not None:
    _nt3 = _g.get("TASK3_NONTRIVIAL_GATES")
    if _nt3 is not None and "task3_circuit_mapping" in _g:
        _m3 = sum(1 for g in _nt3 if task3_circuit_mapping.get(g))
        _notes3 = f"{_m3}/{len(_nt3)} gate types mapped"
    else:
        _sm = _g.get("TASK3_SUMMARY_MAPPED")
        _sn = _g.get("TASK3_SUMMARY_NONTRIVIAL")
        if _sm is not None and _sn is not None:
            _notes3 = f"{_sm}/{_sn} gate types mapped"
        else:
            mapped_count3 = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
            _notes3 = f"{mapped_count3}/{len(non_trivial_gates)} gate types mapped"
    all_results.append({
        'Task': 'Task 3: D&C No Feedback',
        'Method': 'Gate-type mapping, single-shot',
        'SIM_Score': task3_sim_score,
        'Functional': task3_functionally_correct,
        'Overall_Success': bool(task3_functionally_correct and task3_evades),
        'Model': TASK3_MODEL,
        'Notes': _notes3,
    })

# Task 4 Results
if _g.get("task4_final_circuit") is not None and _g.get("TASK4_MODEL") is not None:
    _nt4 = _g.get("TASK4_NONTRIVIAL_GATES")
    if _nt4 is not None and "task4_circuit_mapping" in _g:
        _m4 = sum(1 for g in _nt4 if task4_circuit_mapping.get(g))
        _a4 = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0.0
        _notes4 = f"{_m4}/{len(_nt4)} mapped, {_a4:.1f} avg trials"
    else:
        _m4 = _g.get("TASK4_SUMMARY_MAPPED")
        _n4 = _g.get("TASK4_SUMMARY_NONTRIVIAL")
        _a4 = _g.get("TASK4_SUMMARY_AVG_TRIALS")
        if _m4 is not None and _n4 is not None and _a4 is not None:
            _notes4 = f"{_m4}/{_n4} mapped, {_a4:.1f} avg trials"
        else:
            mapped_count4 = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
            avg_trials4 = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
            _notes4 = f"{mapped_count4}/{len(non_trivial_gates)} mapped, {avg_trials4:.1f} avg trials"
    all_results.append({
        'Task': 'Task 4: D&C With Feedback',
        'Method': 'Gate-type mapping, iterative feedback',
        'SIM_Score': task4_sim_score,
        'Functional': task4_functionally_correct,
        'Overall_Success': bool(task4_functionally_correct and task4_evades),
        'Model': TASK4_MODEL,
        'Notes': _notes4,
    })

# Reference Baseline Results
covered_gates = sum(1 for gt in gate_counts if reference_mapping.get(gt))
all_results.append({
    'Task': f'Reference: {REFERENCE_LLM} Cache',
    'Method': 'Pre-validated research mappings',
    'SIM_Score': ref_sim_score,
    'Functional': ref_equiv,
    'Overall_Success': bool(ref_equiv and ref_evades),
    'Model': REFERENCE_LLM,
    'Notes': f'{covered_gates}/{len(gate_counts)} gate types in cache'
})

# ── Analysis: how to read Yosys vs SIM ──────────────────────
print("\n" + "─" * 60)
print("HOW TO READ THESE METRICS")
print("─" * 60)
_ver = [r["Task"] for r in all_results if r["Functional"] is True]
_fail = [r["Task"] for r in all_results if r["Functional"] is False]
_unk = [r["Task"] for r in all_results if r["Functional"] is None]
print(
    "• Yosys (formal equiv): ground truth for ‘same combinational I/O behavior as gold?’."
)
print(f"  — Equivalent: {', '.join(_ver) if _ver else '— none —'}")
print(f"  — Not equivalent (SAT counterexample / wrong logic): {', '.join(_fail) if _fail else '— none —'}")
print(
    f"  — Unknown (parse / hierarchy / incomplete check): {', '.join(_unk) if _unk else '— none —'}"
)
print(
    "• SIM (text similarity): how different the Verilog *looks*. Low SIM without Yosys ✅ is at best "
    "‘obfuscated or broken’ — not a proven Trojan / replacement."
)
if sys.platform == "darwin":
    print(
        "• Platform: macOS — SIM uses the notebook’s difflib fallback (`sim_text` is Linux-only). "
        "Rank-order is still informative; absolute scores may differ from Colab/Linux."
    )

# ── Print comparison table ──────────────────────────────────
print("\n📋 SUMMARY TABLE:")
print("-" * 60)
for row in all_results:
    print(f"\n{row['Task']}:")
    print(f"  Method: {row['Method']}")
    print(f"  Model:  {row['Model']}")
    print(f"  SIM Score: {_fmt_sim_score(row['SIM_Score'])}")
    func = row['Functional']
    print(f"  Functional: {'✅ YES' if func else ('❌ NO' if func is False else '⚠️ UNKNOWN')}")
    print(f"  Overall Success: {'✅ YES' if row['Overall_Success'] else '❌ NO'}")
    print(f"  Notes: {row['Notes']}")

# ── Ranking: lab definition (overall success = equiv ∧ evades) ──
print(f"\n🎯 RANKING (lab metric: Overall Success, tie-break lower SIM):")
print("-" * 40)

sorted_results = sorted(
    all_results,
    key=lambda r: (int(r['Overall_Success']), -(r['SIM_Score'] if r['SIM_Score'] is not None else 1.0))
)[::-1]

for rank, row in enumerate(sorted_results, 1):
    icon = '🏆' if rank == 1 else '🥈' if rank == 2 else '🥉' if rank == 3 else '📍'
    sim_str = f"SIM={row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "SIM=N/A"
    success_str = '✅' if row['Overall_Success'] else '❌'
    print(f"{rank}. {icon} {row['Task']:35s}  {success_str}  {sim_str}")


def _func_tier(functional):
    if functional is True:
        return 0
    if functional is None:
        return 1
    return 2


sorted_by_yosys = sorted(
    all_results,
    key=lambda r: (
        _func_tier(r["Functional"]),
        r["SIM_Score"] if r["SIM_Score"] is not None else 1.0,
    ),
)
print(f"\n🎯 RANKING (science / report: Yosys ✅ first, then ⚠️, then ❌; lower SIM within tier):")
print("-" * 40)
for rank, row in enumerate(sorted_by_yosys, 1):
    icon = '🏆' if rank == 1 else '🥈' if rank == 2 else '🥉' if rank == 3 else '📍'
    ytag = (
        "Yosys ✅"
        if row["Functional"] is True
        else ("Equiv ⚠️" if row["Functional"] is None else "Yosys ❌")
    )
    sim_str = f"SIM={row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "SIM=N/A"
    print(f"{rank}. {icon} {row['Task']:35s}  {ytag:10s}  {sim_str}")

# ── Key insights ─────────────────────────────────────────────
print(f"\n📈 KEY INSIGHTS:")
valid_sim = [r for r in all_results if r['SIM_Score'] is not None]
if valid_sim:
    verified = [r for r in valid_sim if r['Functional'] is True]
    if verified:
        best_v = min(verified, key=lambda r: r['SIM_Score'])
        print(
            f"• Lowest SIM among Yosys-verified strategies: {best_v['SIM_Score']:.4f}  ({best_v['Task']})"
        )
    else:
        print("• No Yosys-verified strategy in this run — use SIM only as qualitative ‘how different’, not as proof.")
    best_any = min(valid_sim, key=lambda r: r['SIM_Score'])
    if not verified:
        print(
            f"• Lowest SIM overall: {best_any['SIM_Score']:.4f} ({best_any['Task']}) — similarity-only (none equivalent)."
        )
    elif best_any['Functional'] is not True:
        print(
            f"• Lowest SIM overall: {best_any['SIM_Score']:.4f} ({best_any['Task']}) — not Yosys-verified; "
            "low text similarity does not mean a valid or usable replacement."
        )
    if ref_sim_score is not None:
        for row in all_results:
            if row['Task'].startswith('Task') and row['SIM_Score'] is not None:
                diff = row['SIM_Score'] - ref_sim_score
                indicator = '🔺 worse' if diff > 0.02 else ('🔻 better' if diff < -0.02 else '≈ similar')
                print(f"• {row['Task']:35s} vs reference: {diff:+.4f} ({indicator})")

model_results = defaultdict(list)
for r in all_results:
    model_results[r['Model']].append(r['Overall_Success'])
if len(model_results) > 1:
    print(f"\n🤖 MODEL COMPARISON (per strategy row in this table, not per API call):")
    for model, successes in model_results.items():
        rate = sum(successes) / len(successes)
        print(f"• {model}: {rate:.0%} Overall Success (equiv ∧ SIM evasion)")


PART A: REFERENCE BASELINE
🔍 Running Yosys equivalence check for 'sanity_self'...
Sanity check (gold vs same netlist): Circuits formally verified equivalent (Yosys miter+SAT)
Loaded cached mapping: GPT3dot5

Coverage for this circuit's gate types:
  ✅ nand_2: ['AND_NOT', 'OR_NOT']
  ⬜ not_1: no cached mapping (gate kept as-is)
  ✅ xor_2: ['NAND']
  ✅ nor_2: ['AND_NOT']
  ✅ xnor_2: ['NOR']
  ✅ nand_4: ['NOR', 'AND_NOT', 'OR_NOT']
  ⬜ and_9: no cached mapping (gate kept as-is)
  ⬜ and_8: no cached mapping (gate kept as-is)
  ✅ nand_3: ['AND_NOT', 'OR_NOT']

🔄 Applying GPT3dot5 cached mappings (strategy: deterministic)...

🔍 Checking functional equivalence...
🔍 Running Yosys equivalence check for 'reference'...
Functional check: Circuits NOT equivalent — Yosys SAT found a counterexample. Gate-replacement pipelines (cached or LLM) can fail if a mapping is wrong or replacements interact badly. Try another strategy, fix templates, or verify with simulation.

🔍 Evaluating with SIM detector...

## Task 5 — Measured comparison (this workspace)

Use the **Task 5 code cell output** as your numbers (SIM and Yosys depend on circuit, model, and run). On **macOS**, SIM follows the same **difflib** path as `evaluate_sim`; **Yosys** still checks equivalence on the gold/generated Verilog under `tmp_eval`.

**How to use the printed analysis**

- **HOW TO READ THESE METRICS** — **Yosys** = same function as gold; **SIM** = how different the text looks. Those are not interchangeable.
- **Two rankings** — (1) *Lab:* Overall Success = equivalence ∧ SIM below threshold. (2) *Report:* put **Yosys verified** strategies first, then equiv unknown, then not equivalent; compare SIM mainly inside the “verified” group.
- **KEY INSIGHTS** — calls out **lowest SIM among verified strategies** separately from **lowest SIM overall** (which may be a broken or non-equivalent netlist that still looks different).

**Takeaway.** For your write-up, lead with strategies that **pass Yosys**; only then interpret SIM as “stealth” or evasion. A low SIM without equivalence is not a successful replacement — it is at best a failed or invalid rewrite.


## Inference — recorded run (C432 / `c432-CS320`, Gemini 2.5 Pro)

Numbers below match **one full top-to-bottom execution** (macOS, **difflib** SIM proxy). Re-run Tasks **1–5** and refresh this section if you change models, circuits, or seeds.

| Strategy | SIM (↓ stealthier) | Yosys | Lab overall |
|----------|-------------------|-------|-------------|
| Task 1 — Direct Verilog | 0.2185 | **Unknown** (incomplete / no `endmodule` in extract) | Fail |
| Task 2 — Boolean | 0.1250 | **Not equivalent** | Fail |
| Task 3 — D&C, canonical NOR | **0.0147** | **Equivalent** | **Pass** |
| Task 4 — D&C + feedback | 0.2847 | **Not equivalent** | Fail |
| Reference — GPT-3.5 cache | 0.1365 | **Not equivalent** | Fail |

**What to conclude.** Only **Task 3** jointly satisfies **low SIM** and **proved** same I/O behavior. Everything else either **breaks logic** (SAT counterexample), **does not complete** (Task 1), or relies on **partial / mismatched** cached gates (reference).

---

## Conclusions

1. **Text similarity is an incomplete security metric.** Low SIM without Yosys ✅ does not prove a safe or useful rewrite — it may be truncated, syntactically invalid, or functionally wrong.  
2. **Validated transformations dominate naive LLM algebra.** Canonical NOR templates (Task 3) supply a **checkable** mapping per gate type; one-shot or stitched LLM outputs do not.  
3. **Feedback without formal checks stays brittle.** Task 4’s loop mostly repairs **format**; bad Boolean structure still passes until **full-circuit** equivalence fails.  
4. **Measurement environment matters.** On macOS, SIM here is **difflib** unless you use Linux/`sim_text`; **Yosys** remains the authority for equivalence.

**Bottom line.** For this benchmark and toolchain, **Task 3 (deterministic NOR, divide-and-conquer)** is the only approach in the recorded run that meets both **SIM \< 0.3** and **formal equivalence**.
